# HotelIQ – Model Building and Evaluation

## Objective

The objective of this notebook is to develop machine learning models that predict hotel booking cancellations.

### Steps Performed

- Load the cleaned hotel booking dataset.
- Separate independent and dependent variables.
- Split the dataset into training and testing sets.
- Apply feature preprocessing.
- Train multiple classification algorithms.
- Evaluate model performance using classification metrics.
- Compare models and identify the best-performing model.

### Machine Learning Algorithms

1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. XGBoost Classifier

In [44]:
# Importing required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [45]:
df = pd.read_csv('../data/hotel_bookings_cleaned.csv')

df.shape

(87395, 31)

In [46]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,-1.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [47]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 87395 entries, 0 to 87394
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87395 non-null  str    
 1   is_canceled                     87395 non-null  int64  
 2   lead_time                       87395 non-null  int64  
 3   arrival_date_year               87395 non-null  int64  
 4   arrival_date_month              87395 non-null  str    
 5   arrival_date_week_number        87395 non-null  int64  
 6   arrival_date_day_of_month       87395 non-null  int64  
 7   stays_in_weekend_nights         87395 non-null  int64  
 8   stays_in_week_nights            87395 non-null  int64  
 9   adults                          87395 non-null  int64  
 10  children                        87395 non-null  float64
 11  babies                          87395 non-null  int64  
 12  meal                            87395 non-n

### Separating Features and Target

The target variable is `is_canceled`, which indicates whether a hotel booking was cancelled.

- **X:** Independent features used to predict booking cancellation.
- **y:** Target variable that the model will predict.

Columns that may cause data leakage or are unavailable at the time of booking are excluded.

In [48]:
# columns to exclude

drop_columns = ['is_canceled', 'reservation_status',
                'reservation_status_date', 'booking_changes', 
                'assigned_room_type', 'days_in_waiting_list']

#separating features and target
X = df.drop(columns=drop_columns)
y = df['is_canceled']

print("Features Shape: ", X.shape)
print("Target Shape: ", y.shape)



Features Shape:  (87395, 25)
Target Shape:  (87395,)


### Train-Test Split

The dataset is divided into training and testing sets.

- 80% of the data is used for training.
- 20% of the data is used for testing.
- Stratified sampling maintains the target class distribution.
- Random state is set to 49 to ensure reproducibility and consistency with the preprocessing notebook.

In [49]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test,  = train_test_split(X,y, test_size=0.20,random_state=49, stratify=y)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training Features: (69916, 25)
Testing Features: (17479, 25)

Training Target Distribution:
is_canceled
0    72.509869
1    27.490131
Name: proportion, dtype: float64

Testing Target Distribution:
is_canceled
0    72.509869
1    27.490131
Name: proportion, dtype: float64


### Identifying Numerical and Categorical Features

The independent features are divided into numerical and categorical groups.

- Numerical features will be standardized using StandardScaler.
- Categorical features will be encoded using OneHotEncoder.

The `agent` column is treated as categorical because it represents an agent identifier rather than a continuous numerical measurement.

In [50]:
# Define categorical features
categorical_features = ['hotel', 'arrival_date_month', 'meal', 
                        'country', 'market_segment', 
                        'distribution_channel', 'reserved_room_type', 
                        'deposit_type', 'customer_type', 'agent']

# Define numerical features
numerical_features = [col for col in X.columns
                      if col not in categorical_features]

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

Numerical Features: 15
Categorical Features: 10


In [51]:
# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

print("Preprocessor created successfully!")

Preprocessor created successfully!


### 1. Logistic Regression

Logistic Regression is used as the baseline classification model for predicting hotel booking cancellations.

It estimates the probability of a booking cancellation based on the selected independent features.

The preprocessing transformer and Logistic Regression model are combined using a Scikit-learn Pipeline.

In [52]:
# creating logistic regression pipeline

logistic_model = Pipeline(steps=[('preprocessor',preprocessor),
                          ('classifier',LogisticRegression(max_iter=1000, random_state=49))])

print("Logistics Regression pipeline created successfully!")


Logistics Regression pipeline created successfully!


In [53]:
#training model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


### Logistic Regression Model Evaluation

The trained Logistic Regression model is evaluated using the testing dataset.

Predictions are generated to compare the model's predicted outcomes with the actual booking cancellation status.

In [54]:
# Generate predictions
y_pred_logistic = logistic_model.predict(X_test)

# Generate prediction probabilities
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_logistic[:10])

Predictions generated successfully!

First 10 Actual Values:
[0 0 0 0 0 0 1 1 0 0]

First 10 Predicted Values:
[0 0 0 0 0 0 1 0 0 0]


In [55]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report)

#### Classification Performance Metrics

The Logistic Regression model is evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC score

In [56]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred_logistic)
precision = precision_score(y_test, y_pred_logistic)
recall = recall_score(y_test, y_pred_logistic)
f1 = f1_score(y_test, y_pred_logistic)
roc_auc = roc_auc_score(y_test, y_prob_logistic)

# Display results
print("Logistic Regression Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

Logistic Regression Performance
--------------------------------
Accuracy:  0.7974
Precision: 0.6730
Recall:    0.5113
F1 Score:  0.5811
ROC-AUC:   0.8553


#### Confusion Matrix

A confusion matrix helps us understand the number of correct and incorrect predictions made by the Logistic Regression model.

It shows:
- True Positives (TP)
- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)

In [57]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_logistic)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[11480  1194]
 [ 2348  2457]]


#### Classification Report

The classification report provides precision, recall, F1-score and support for both classes.

It helps evaluate how effectively the model predicts canceled and non-canceled bookings.

In [58]:
print(classification_report(y_test, y_pred_logistic))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87     12674
           1       0.67      0.51      0.58      4805

    accuracy                           0.80     17479
   macro avg       0.75      0.71      0.72     17479
weighted avg       0.79      0.80      0.79     17479



### 1. Logistic Regression – Overall Findings

The Logistic Regression model achieved an accuracy of 79.74% and an ROC-AUC score of 85.53%.

The model performs better at identifying non-canceled bookings than canceled bookings.

The recall for canceled bookings is 51.13%, indicating that the model misses a considerable number of actual cancellations.

Logistic Regression provides a useful baseline model. Other classification algorithms will be evaluated to determine whether cancellation prediction performance can be improved.

## 2. Decision Tree Classifier

A Decision Tree Classifier is a supervised machine learning algorithm that makes predictions using a tree-like structure of decision rules.

It splits the dataset based on feature values to classify hotel bookings as canceled or not canceled.

The model will use the same preprocessing steps and training dataset as Logistic Regression.

In [59]:
# create decision tree pipeline

decision_tree_model = Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', DecisionTreeClassifier(random_state=49))])

#training model
decision_tree_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


#### Decision Tree Predictions

The trained Decision Tree model is used to predict whether hotel bookings in the testing dataset will be canceled or not.

Prediction probabilities are also generated for calculating the ROC-AUC score.

In [60]:
# Generate predictions
y_pred_dt = decision_tree_model.predict(X_test)

# Generate prediction probabilities
y_prob_dt = decision_tree_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_dt[:10])

Predictions generated successfully!

First 10 Actual Values:
[0 0 0 0 0 0 1 1 0 0]

First 10 Predicted Values:
[0 1 1 0 1 0 1 0 0 0]


#### Decision Tree – Performance Evaluation

The Decision Tree model is evaluated using accuracy, precision, recall, F1-score and ROC-AUC.

These metrics help measure how effectively the model identifies canceled and non-canceled hotel bookings.

In [61]:
# Calculate evaluation metrics
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

# Display results
print("Decision Tree Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall:    {recall_dt:.4f}")
print(f"F1 Score:  {f1_dt:.4f}")
print(f"ROC-AUC:   {roc_auc_dt:.4f}")

Decision Tree Performance
--------------------------------
Accuracy:  0.8039
Precision: 0.6434
Recall:    0.6429
F1 Score:  0.6431
ROC-AUC:   0.7559


In [62]:
# Generate confusion matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Decision Tree Confusion Matrix:")
print(cm_dt)

Decision Tree Confusion Matrix:
[[10962  1712]
 [ 1716  3089]]


In [63]:
#Decision Tree – Classification Report

#The classification report evaluates the Decision Tree model using precision, 
# recall, F1-score and support for both target classes.

print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86     12674
           1       0.64      0.64      0.64      4805

    accuracy                           0.80     17479
   macro avg       0.75      0.75      0.75     17479
weighted avg       0.80      0.80      0.80     17479



### 2. Decision Tree – Overall Findings

The Decision Tree model achieved an accuracy of 80.39% and an ROC-AUC score of 75.59%.

Compared with Logistic Regression, the Decision Tree model achieved better recall and F1-score for canceled bookings.

It identified 64.29% of actual cancellations, compared with 51.13% for Logistic Regression.

However, its ROC-AUC score was lower, indicating weaker overall discrimination between canceled and non-canceled bookings.

The Decision Tree model provides a more balanced classification of the two classes, but further models will be evaluated to identify the best-performing approach.

## 3. Random Forest Classifier

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to make predictions.

Each tree is trained using a different sample of the training data, and their predictions are combined to produce the final classification.

Random Forest generally helps reduce overfitting compared with an individual Decision Tree.

In [64]:
# create random forest pipeline

random_forest_model = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('classifier', RandomForestClassifier(random_state=49, 
                                                                            n_estimators=100, n_jobs=-1))])

#train model
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


#### Random Forest Predictions

The trained Random Forest model is used to predict whether hotel bookings in the testing dataset will be canceled or not.

Prediction probabilities are also generated to evaluate the model using the ROC-AUC score.

In [65]:
# Generate predictions
y_pred_rf = random_forest_model.predict(X_test)

# Generate prediction probabilities
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_rf[:10])

Predictions generated successfully!

First 10 Actual Values:
[0 0 0 0 0 0 1 1 0 0]

First 10 Predicted Values:
[0 0 0 0 0 0 0 0 0 0]


#### Random Forest – Performance Evaluation

The Random Forest model is evaluated using accuracy, precision, recall, F1-score and ROC-AUC.

These metrics help assess how effectively the model identifies canceled and non-canceled hotel bookings.

In [66]:
# Calculate evaluation metrics
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

# Display results
print("Random Forest Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall:    {recall_rf:.4f}")
print(f"F1 Score:  {f1_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_rf:.4f}")

Random Forest Performance
--------------------------------
Accuracy:  0.8471
Precision: 0.7713
Recall:    0.6310
F1 Score:  0.6941
ROC-AUC:   0.9032


In [67]:
# confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Confusion Matrix:")
print(cm_rf)

Random Forest Confusion Matrix:
[[11775   899]
 [ 1773  3032]]


In [68]:
# classifcation report
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90     12674
           1       0.77      0.63      0.69      4805

    accuracy                           0.85     17479
   macro avg       0.82      0.78      0.80     17479
weighted avg       0.84      0.85      0.84     17479



### Random Forest – Overall Findings

The Random Forest model achieved an accuracy of 84.71% and an ROC-AUC score of 90.32%.

The model performed well in identifying non-canceled bookings, achieving a recall of 93%.

For canceled bookings, the model achieved a precision of 77.13% and a recall of 63.10%, indicating that it correctly identifies approximately 63% of actual cancellations.

Compared with Logistic Regression and Decision Tree, Random Forest achieved the highest accuracy, precision, F1-score, and ROC-AUC so far.

Overall, Random Forest demonstrates strong predictive performance and is currently the best-performing model among the three algorithms evaluated.

## 4. XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is an ensemble learning algorithm that builds multiple decision trees sequentially.

Each new tree tries to correct the errors made by the previous trees. This helps improve the model's predictive performance.

In [69]:
from xgboost import XGBClassifier

In [70]:
#The model will use 100 boosting rounds with a learning rate of 0.1 
#and a maximum tree depth of 6.

xgb_model = Pipeline(steps=[("preprocessor", preprocessor),
                            ('classifier', XGBClassifier(n_estimators=100,
                                                         learning_rate=0.1,
                                                         max_depth=6,
                                                         random_state=49,
                                                         eval_metric='logloss',
                                                         n_jobs=-1))])

#train model
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

XGBoost model trained successfully!


In [71]:
# Predict class labels
y_pred_xgb = xgb_model.predict(X_test)

# Predict cancellation probabilities
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("Actual Values:     ", y_test[:10].values)
print("Predicted Values:  ", y_pred_xgb[:10])

Actual Values:      [0 0 0 0 0 0 1 1 0 0]
Predicted Values:   [0 0 0 0 0 0 0 0 0 0]


In [72]:
# Calculate evaluation metrics

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

precision_xgb = precision_score(y_test, y_pred_xgb)

recall_xgb = recall_score(y_test, y_pred_xgb)

f1_xgb = f1_score(y_test, y_pred_xgb)

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)


print("XGBoost Model Performance")
print("-------------------------")
print(f"Accuracy:  {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall:    {recall_xgb:.4f}")
print(f"F1 Score:  {f1_xgb:.4f}")
print(f"ROC-AUC:   {roc_auc_xgb:.4f}")

XGBoost Model Performance
-------------------------
Accuracy:  0.8372
Precision: 0.7421
Recall:    0.6252
F1 Score:  0.6786
ROC-AUC:   0.9030


In [73]:
# confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("XGBoost Confusion Matrix:")
print(cm_xgb)

XGBoost Confusion Matrix:
[[11630  1044]
 [ 1801  3004]]


In [74]:
# classification report

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89     12674
           1       0.74      0.63      0.68      4805

    accuracy                           0.84     17479
   macro avg       0.80      0.77      0.78     17479
weighted avg       0.83      0.84      0.83     17479



### XGBoost – Overall Findings

The XGBoost model achieved an accuracy of **83.72%** and an ROC-AUC score of **90.30%**.

For non-canceled bookings (Class 0), the model achieved a precision of 87%, recall of 92%, and an F1-score of 89%.

For canceled bookings (Class 1), the model achieved a precision of 74%, recall of 63%, and an F1-score of 68%.

The confusion matrix shows that the model correctly identified 3,004 canceled bookings and 11,630 non-canceled bookings. However, it missed 1,801 actual cancellations.

The model demonstrates strong overall predictive performance and good ability to distinguish between canceled and non-canceled bookings.

Compared with the Random Forest model, XGBoost achieved slightly lower accuracy, precision, recall, F1-score, and ROC-AUC.

Overall, XGBoost is a strong classification model, but Random Forest currently performs better across all five evaluation metrics.

## Model Comparison

In this section, we compare the performance of all four classification models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

The models will be compared using five evaluation metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

This comparison will help us identify the most suitable model for predicting hotel booking cancellations.

In [75]:
model_comparison = pd.DataFrame({'Model':['Logistic Regression', 'Decision Tree','Random Forest','XGBoost'],
                                 'Accuracy':[accuracy, accuracy_dt,accuracy_rf,accuracy_xgb],
                                 'Precision':[precision,precision_dt,precision_rf,precision_xgb],
                                 'Recall':[recall,recall_dt,recall_rf,recall_xgb],
                                 'F1 Score':[f1, f1_dt,f1_rf,f1_xgb],
                                 'ROC AUC':[roc_auc,roc_auc_dt,roc_auc_rf,roc_auc_xgb]})

model_comparison = model_comparison.sort_values(by="ROC AUC",ascending=False).reset_index(drop=True)

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Random Forest,0.847131,0.771305,0.631009,0.694139,0.903209
1,XGBoost,0.837233,0.742095,0.625182,0.678640,0.903027
2,Logistic Regression,0.797357,0.672966,0.511342,0.581126,0.855283
3,Decision Tree,0.803879,0.643408,0.642872,0.643140,0.755854


### Model Comparison – Overall Findings

Four classification algorithms were evaluated for hotel booking cancellation prediction.

The Random Forest model achieved the best overall performance, with an accuracy of 84.71%, precision of 77.13%, recall of 63.10%, F1-score of 69.41%, and ROC-AUC of 90.32%.

XGBoost also demonstrated strong performance, achieving an ROC-AUC score of 90.30%, which is very close to Random Forest.

The Decision Tree model achieved the highest recall of 64.29%, making it slightly better at identifying actual cancellations. However, its overall ROC-AUC score was lower.

Logistic Regression served as the baseline model but showed comparatively lower predictive performance.

Based on the overall evaluation metrics, Random Forest is selected as the best-performing model for further investigation.

However, the final model selection will also consider the business objective of identifying canceled bookings and the importance of reducing missed cancellations.

## Random Forest – Threshold Tuning

### Objective

The objective of threshold tuning is to identify a suitable classification threshold that improves the model's ability to identify canceled bookings.

By default, binary classification models use a threshold of 0.50.

We will experiment with different thresholds using a separate validation dataset and evaluate the selected threshold on the untouched test dataset.

In [76]:
# Split training data into training and validation sets

X_train_new, X_val, y_train_new, y_val = train_test_split(X_train,y_train,test_size=0.20,
                                                          random_state=49,stratify=y_train)

print("Training Data Shape:", X_train_new.shape)
print("Validation Data Shape:", X_val.shape)

Training Data Shape: (55932, 25)
Validation Data Shape: (13984, 25)


In [77]:
# Create Random Forest pipeline

rf_tuning_model = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('classifier', RandomForestClassifier(n_estimators=100,
                                                                        random_state=49,
                                                                        n_jobs=-1))])

In [78]:

rf_tuning_model.fit(X_train_new, y_train_new)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](25,)","['hotel','lead_time','arrival_date_year',...,'adr', 'required_car_parking_spaces','total_of_special_requests']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,25
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By 

In [79]:
# Generate cancellation probabilities

y_val_prob = rf_tuning_model.predict_proba(X_val)[:, 1]

# Display first 10 probabilities

print("First 10 Cancellation Probabilities:")
print(y_val_prob[:10])

First 10 Cancellation Probabilities:
[0.18  0.42  0.09  0.76  0.575 0.41  0.68  0.68  0.12  0.05 ]


In [80]:
# Define classification thresholds

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:

    # Convert probabilities into predictions
    y_val_pred = (y_val_prob >= threshold).astype(int)

    # Calculate evaluation metrics
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    threshold_results.append({'Threshold': threshold,'Precision': precision,
                              'Recall': recall,'F1 Score': f1})

# Create comparison DataFrame

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.30,0.617239,0.832726,0.708970
1,0.35,0.661778,0.786420,0.718735
2,0.40,0.704596,0.737773,0.720803
3,0.45,0.735064,0.684964,0.709130
4,0.50,0.765264,0.629292,0.690650
5,0.55,0.798624,0.573621,0.667676
6,0.60,0.824725,0.506764,0.627780


### Threshold Tuning – Overall Findings

The Random Forest model was evaluated using seven different classification thresholds ranging from 0.30 to 0.60.

The results showed that lowering the classification threshold increased recall while reducing precision. Increasing the threshold improved precision but reduced recall.

Among the tested thresholds, 0.40 achieved the highest F1-score of 72.08%, with a precision of 70.46% and recall of 73.78%.

The threshold of 0.30 achieved the highest recall of 83.27%, making it suitable when identifying as many actual cancellations as possible is the primary objective.

Based on the balance between precision and recall, a threshold of 0.40 is selected as the initial candidate for further evaluation.

The selected threshold will be evaluated on the untouched test dataset before finalizing the model.

### Final Random Forest Model

After selecting 0.40 as the initial classification threshold using validation data, we will retrain the Random Forest model using the complete training dataset.

The test dataset will remain untouched and will be used only for final model evaluation.

In [81]:
# Create the final Random Forest pipeline

rf_final_model = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', RandomForestClassifier(n_estimators=100,
                                                                       random_state=49,
                                                                       n_jobs=-1))])

# Train the model on the complete training dataset

rf_final_model.fit(X_train, y_train)

print("Final Random Forest Model trained successfully!")

Final Random Forest Model trained successfully!


In [82]:
# Generate cancellation probabilities

y_test_prob_rf = rf_final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold

best_threshold = 0.40

y_test_pred_rf_tuned = (y_test_prob_rf >= best_threshold).astype(int)

print("First 10 Cancellation Probabilities:")
print(y_test_prob_rf[:10])

print("\nFirst 10 Predictions:")
print(y_test_pred_rf_tuned[:10])

First 10 Cancellation Probabilities:
[0.   0.2  0.24 0.01 0.29 0.01 0.2  0.13 0.22 0.2 ]

First 10 Predictions:
[0 0 0 0 0 0 0 0 0 0]


### Final Model Evaluation

We will evaluate the final Random Forest model on the unseen test dataset using the following metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

These metrics will help us understand how effectively the model identifies hotel booking cancellations.

In [83]:
# Calculate evaluation metrics

rf_tuned_accuracy = accuracy_score(y_test, y_test_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_test_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_test_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_test_pred_rf_tuned)
rf_tuned_roc_auc = roc_auc_score(y_test, y_test_prob_rf)

# Display results

print("Random Forest - Tuned Threshold (0.40)")
print("---------------------------------------")
print(f"Accuracy:  {rf_tuned_accuracy:.4f}")
print(f"Precision: {rf_tuned_precision:.4f}")
print(f"Recall:    {rf_tuned_recall:.4f}")
print(f"F1 Score:  {rf_tuned_f1:.4f}")
print(f"ROC-AUC:   {rf_tuned_roc_auc:.4f}")

Random Forest - Tuned Threshold (0.40)
---------------------------------------
Accuracy:  0.8430
Precision: 0.7013
Recall:    0.7471
F1 Score:  0.7235
ROC-AUC:   0.9032


### Final Model Evaluation – Findings

The Random Forest model was evaluated on the unseen test dataset using the selected classification threshold of 0.40.

The model achieved:

- Accuracy: 84.30%
- Precision: 70.13%
- Recall: 74.71%
- F1-score: 72.35%
- ROC-AUC: 90.32%

Compared with the original Random Forest model using the default threshold of 0.50, the tuned model achieved higher recall and F1-score.

Recall increased from 63.10% to 74.71%, indicating that the model can identify a larger proportion of actual booking cancellations.

However, precision decreased from 77.13% to 70.13%, indicating an increase in false cancellation predictions.

Based on these results, the threshold of 0.40 provides a better balance between identifying cancellations and maintaining precision for this experiment.

In [84]:
# Generate confusion matrix

cm_rf_tuned = confusion_matrix(y_test, y_test_pred_rf_tuned)

print("Confusion Matrix:")
print(cm_rf_tuned)

Confusion Matrix:
[[11145  1529]
 [ 1215  3590]]


## Final Model Selection

After evaluating Logistic Regression, Decision Tree, Random Forest, and XGBoost, the Random Forest Classifier was selected for further development.

The Random Forest model achieved the highest ROC-AUC of 90.32% and the highest F1-score among the four models at the default classification threshold.

To improve cancellation detection, threshold tuning was performed. A classification threshold of 0.40 was selected based on the highest validation F1-score.

On the unseen test dataset, the tuned Random Forest model achieved:

- Accuracy: 84.30%
- Precision: 70.13%
- Recall: 74.71%
- F1-score: 72.35%
- ROC-AUC: 90.32%

The tuned model demonstrated improved recall and F1-score compared with the original Random Forest model.

Therefore, Random Forest with a classification threshold of 0.40 was selected as the final model candidate for the HotelIQ project.

In [85]:
import os
import joblib

# Creating models folder 
os.makedirs("../models", exist_ok=True)

# Save trained model and threshold
model_artifact = {"model": rf_final_model,"threshold": 0.40}

joblib.dump(model_artifact,"../models/hotel_cancellation_model.pkl")

print("Model saved successfully!")

Model saved successfully!
